# D1-A2-R1 — YOLO11s-54 repeat-factor train

Use a new NVIDIA L4 runtime for each leg. Run `fresh` to epoch 1, stop for independent live-smoke QA, then change `MODE` to `resume` on a distinct L4 runtime. Do not open sealed test. The Git checkout remains clean.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
import hashlib
import json
import re
import shutil
import subprocess
import sys
from pathlib import Path

MODE = "fresh"  # Switch to "resume" only after independent live-smoke QA PASS.
CODE_COMMIT = "f3f7571794a5db04ad302fecf8014546ec1eb255"
RUN_LABEL = "d1_a2_seed_20261018_r1"
DRIVE_ROOT = Path("/content/drive/MyDrive/CPV301/Traffic_Sign_Detector/M1_D1_Colab")
DRIVE_BUNDLE = DRIVE_ROOT / "m1_colab_train_val_backend_r1.tar.gz"
SIDECAR = DRIVE_ROOT / "m1_colab_train_val_backend_r1.tar.gz.manifest.json"
STATE_DIR = DRIVE_ROOT / "runs" / RUN_LABEL
REPO_DIR = Path("/content/Traffic_Sign_Detector")
REPO_URL = "https://github.com/ScottPhantom/Traffic_SIgn_Detection.git"
OWNER_AUTHORIZATION = "docs/review/d1_a2_r1_owner_authorization_2026-11-05.md"
OWNER_AUTHORIZATION_SHA256 = (
    "1a11b283da34848dc873b577b42240a5182b795654f5a3db9349369ae78221c9"
)
TRAIN_LIST_SHA256 = "90d735d67bde2bf736b5fc8570110561f0184747f3fb25c8f42a3a599d0fdec2"
ROSTER_SHA256 = "0ce6afdc84ffab0db361920c4c7817517353fd8b1b901e51770d2f3f4eb69aa3"
EXPECTED_BUNDLE_BYTES = 766226800
EXPECTED_BUNDLE_SHA256 = "7ea344cb1e2c166786d0552f215993969efec9b5f06a9bf49310d4390243d641"
EXPECTED_SIDECAR_BYTES = 1499866
EXPECTED_SIDECAR_SHA256 = "7e0dc5b0144faa5f0be407224eaf8070dd51199fa8a1d2f164b248fd6a06d5b2"
PART_SPECS = (
    (
        'm1_colab_train_val_backend_r1.tar.gz.part-00',
        99614720,
        '53f30f08406fbef5b8451610b2fd44a1f2be42ca6d50f612b68e5ace285e66ff',
    ),
    (
        'm1_colab_train_val_backend_r1.tar.gz.part-01',
        99614720,
        '8451fe932760179d4b2e1a9cbdd21fd58744bc6b8f3d2a90e431cece7f546544',
    ),
    (
        'm1_colab_train_val_backend_r1.tar.gz.part-02',
        99614720,
        '867c935bf152ddfc3e7919547dccf3682291326b60b956a8100fc1a59b60e98d',
    ),
    (
        'm1_colab_train_val_backend_r1.tar.gz.part-03',
        99614720,
        '046fc8ae5d4e0a91196ccd6a6afa422fb1254baea7ae30bc3cab39602b534907',
    ),
    (
        'm1_colab_train_val_backend_r1.tar.gz.part-04',
        99614720,
        '6d3751d858232fada5d57e8e892f3e272cd5b9e34efb184971b3f4786842665d',
    ),
    (
        'm1_colab_train_val_backend_r1.tar.gz.part-05',
        99614720,
        '938106e4d34743e9b79109feb98d573f317afad44289f792f070f22e3924159b',
    ),
    (
        'm1_colab_train_val_backend_r1.tar.gz.part-06',
        99614720,
        '3497c1a10314c74d4b81ba9a5a49653b16ba2e2d52b424fd7655623547cb6408',
    ),
    (
        'm1_colab_train_val_backend_r1.tar.gz.part-07',
        68923760,
        '5f5ca85bc392310b2b5693ad9d0723f046eca9fd7980b45c944f22a3b3bc0eea',
    ),
)

assert MODE in {"fresh", "resume"}
assert re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT)


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


In [ ]:
assert SIDECAR.is_file() and SIDECAR.stat().st_size == EXPECTED_SIDECAR_BYTES
assert sha256_file(SIDECAR) == EXPECTED_SIDECAR_SHA256
if DRIVE_BUNDLE.is_file():
    BUNDLE = DRIVE_BUNDLE
    bundle_source = "single_archive"
else:
    parts = []
    for name, expected_bytes, expected_sha256 in PART_SPECS:
        part = DRIVE_ROOT / name
        assert part.is_file() and part.stat().st_size == expected_bytes, part
        assert sha256_file(part) == expected_sha256, part
        parts.append(part)
    BUNDLE = Path("/content/m1_colab_train_val_backend_r1.tar.gz")
    assert not BUNDLE.exists(), "Refusing to reuse an existing assembled bundle"
    with BUNDLE.open("xb") as destination:
        for part in parts:
            with part.open("rb") as source:
                for chunk in iter(lambda: source.read(1024 * 1024), b""):
                    destination.write(chunk)
    bundle_source = "verified_drive_parts"
assert BUNDLE.stat().st_size == EXPECTED_BUNDLE_BYTES
assert sha256_file(BUNDLE) == EXPECTED_BUNDLE_SHA256
if MODE == "fresh":
    assert not STATE_DIR.exists(), f"Fresh state already exists: {STATE_DIR}"
else:
    assert (STATE_DIR / "recovery/latest.json").is_file(), "Recovery pointer missing"
print({"bundle": "PASS", "source": bundle_source, "mode": MODE})


In [ ]:
gpu_name = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
assert gpu_name == "NVIDIA L4", gpu_name
assert not REPO_DIR.exists(), f"Use a fresh runtime; repo directory exists: {REPO_DIR}"
subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "checkout", "--detach", CODE_COMMIT], cwd=REPO_DIR, check=True)
observed_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
assert observed_commit == CODE_COMMIT
assert not subprocess.run(
    ["git", "status", "--porcelain", "--untracked-files=all"],
    cwd=REPO_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip(), "A2 requires a clean checkout"
authorization_path = REPO_DIR / OWNER_AUTHORIZATION
assert authorization_path.is_file() and not authorization_path.is_symlink()
assert sha256_file(authorization_path) == OWNER_AUTHORIZATION_SHA256
if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
print({"gpu": gpu_name, "checkout": observed_commit, "authorization": "PASS"})


In [ ]:
command = [
    sys.executable,
    str(REPO_DIR / "detector/colab/run_d1_colab_a2_r1.py"),
    "--repo-root", str(REPO_DIR),
    "--bundle", str(BUNDLE),
    "--state-dir", str(STATE_DIR),
    "--mode", MODE,
    "--device", "0",
    "--owner-authorization", OWNER_AUTHORIZATION,
    "--owner-authorization-sha256", OWNER_AUTHORIZATION_SHA256,
    "--expected-checkout", CODE_COMMIT,
    "--download-approved-weight",
]
if MODE == "fresh":
    command.extend(["--stop-after-epoch", "1"])
print("Running:", " ".join(command))
subprocess.run(command, check=True)


In [ ]:
execution = json.loads((STATE_DIR / "colab_execution.json").read_text())
latest = json.loads((STATE_DIR / "recovery/latest.json").read_text())
snapshot = STATE_DIR / "recovery/snapshots" / latest["generation"]
run_contract = json.loads((snapshot / "run_contract.json").read_text())
assert execution["experiment"] == "D1-A2-R1"
assert execution["package"] == "D1-A2-R1"
assert execution["sealed_test_accessed"] is False
assert execution["checkout_commit"] == CODE_COMMIT
assert execution["owner_authorization"]["sha256"] == OWNER_AUTHORIZATION_SHA256
assert run_contract["environment"]["git_dirty_files"] == 0
assert run_contract["environment"]["git_commit"] == CODE_COMMIT
assert run_contract["a2_data"]["train_list_sha256"] == TRAIN_LIST_SHA256
assert run_contract["a2_data"]["repeated_roster_sha256"] == ROSTER_SHA256
assert run_contract["a2_data"]["train_list_entries"] == 4687
assert run_contract["owner_authorization"]["sha256"] == OWNER_AUTHORIZATION_SHA256
assert run_contract["sealed_test_accessed"] is False
assert "4687" in (STATE_DIR / "colab_train.log").read_text()
if MODE == "fresh":
    assert execution["status"] == "PAUSED_FOR_RESUME", execution
    assert latest["generation"] == "epoch-0001", latest
    print("A2 epoch 1 verified. Stop for independent live-smoke QA.")
else:
    assert execution["status"] == "SUCCEEDED", execution
    assert execution["training_outcome"]["status"] in {
        "COMPLETED", "COMPLETED_FROM_FINAL_CHECKPOINT"
    }
    assert execution["training_outcome"]["completed_epoch"] == 100
    assert latest["generation"] == "epoch-0100", latest
    archives = sorted(STATE_DIR.glob("*_result.tar.gz"))
    assert len(archives) == 1, archives
    manifest = archives[0].with_name(f"{archives[0].name}.manifest.json")
    assert manifest.is_file()
    print({"a2_epoch_100": "PASS", "archive": str(archives[0])})
